In [1]:
# imports
import os, sys, time
from tqdm.notebook import tqdm

# import the __functions.py (custom functions)
sys.path.append(os.getcwd()) # add code folder to system path
from __functions import *  # imports all custom functions

# local data directories
datadir = '/Users/mcc/Library/CloudStorage/Box-Box/MCC/data/'
projdir = os.path.dirname(os.getcwd())
print(projdir)

print("Complete!")

/Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation
Complete!


In [2]:
# read in the incidents perimeter data
fp = os.path.join(projdir, 'data/spatial/mod/ics209plus_2014to2023_perimeters_tcc.gpkg')
incis = gpd.read_file(fp)
print(incis.head())

  CAUSE COMPLEX      DISCOVERY_DATE  DISCOVERY_DOY EVACUATION_REPORTED  \
0     H   False 2021-05-24 15:48:00          144.0                None   
1     H   False 2015-08-28 00:13:00          240.0                None   
2     L   False 2022-05-30 01:10:00          150.0                None   
3     L   False 2015-06-22 16:49:00          173.0                None   
4     L   False 2015-06-20 22:19:00          171.0                None   

  EXPECTED_CONTAINMENT_DATE  FATALITIES  FATALITIES_PUBLIC  \
0                      None         0.0                NaN   
1                      None         0.0                0.0   
2                      None         0.0                NaN   
3                      None         0.0                NaN   
4                      None         0.0                NaN   

   FATALITIES_RESPONDER  FINAL_ACRES  ... date_diff_days match_key  IRWIN_C  \
0                   NaN       3750.0  ...            0.0      None     None   
1                   0.0 

In [3]:
# --- Check on destructive incidents
print(len(incis))
print(len(incis[incis['STR_THREATENED_MAX']>0]))

6940
3293


In [4]:
# --- Keep only forested incidents
incis_ = incis.copy()
incis_ = incis_[
    (incis_['FUEL_MODEL'].str.contains('Timber', case=False)) |
    (incis_['tcc']>=10) # keep forested incidents
]
print(len(incis_))

4259


In [5]:
# Load counties and join to get list of FIPS
counties = os.path.join(datadir,"boundaries/political/TIGER/tl_2024_us_county/tl_2024_us_county.shp")
counties = gpd.read_file(counties).to_crs(incis.crs)

# Intersect with fires
fire_fips = gpd.sjoin(counties, incis_, how="inner", predicate="intersects")
unique_fips = fire_fips["GEOID"].unique()
print(f"{len(unique_fips)} counties intersect with fire perimeters.\n")

# create a lookup table for fires
# Group by Fire ID and collect FIPS codes (GEOID)
fire_fips_lookup = (
    fire_fips.groupby("INCIDENT_ID")["GEOID"]
    .unique()  # get unique FIPS per fire
    .reset_index()
)
# save a table for later
out_fp = os.path.join(projdir,"data/tabular/fire_to_fips.csv")
fire_fips_lookup.to_csv(out_fp, index=False)
print(f"Saved to {out_fp}")

766 counties intersect with fire perimeters.

Saved to /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/tabular/fire_to_fips.csv


### Access the NSI by county FIPS code


In [7]:
# --- Search for already downloaded FIPS data
print(len(unique_fips))
out_fp = os.path.join(projdir,"data/spatial/nsi_structures_in_fires.geojson")
fips_c = []
if not os.path.exists(out_fp):
    print("No structure data found")
else:
    nsi = gpd.read_file(out_fp)
    fips_c = nsi['FIPS'].unique()
    del nsi
print(f"Counties already downloaded: {len(fips_c)}")

# modify the list if needed...
if len(fips_c)>0:
    unique_fips = list(set(unique_fips) - set(fips_c))
print(f"Remaining FIPS: {len(unique_fips)}")

766
Counties already downloaded: 570
Remaining FIPS: 311


In [8]:
# Download the NSI data for all FIPS
# see __functions.py
nsi_results, failed_fips = fetch_nsi_fips(unique_fips, timeout=180)

# Retry on failed downloads
max_retries = 3
retry_count = 0
while failed_fips and retry_count < max_retries:
    print(f"\nRetrying {len(failed_fips)} failed FIPS (Attempt {retry_count + 1})")
    time.sleep(5)  # Optional small pause between rounds
    retry_results, failed_fips = fetch_nsi_fips(failed_fips, timeout=220)
    nsi_results.extend(retry_results)
    retry_count += 1

if failed_fips:
    print(f"Final failed FIPS after {max_retries} retries: {failed_fips}")
    # Optional: save to file
    with open("failed_fips.txt", "w") as f:
        for fips in failed_fips:
            f.write(f"{fips}\n")
else:
    print("All FIPS successfully downloaded.")

No structures returned for FIPS 46102
No structures returned for FIPS 02158
No structures returned for FIPS 02066

Retrying 3 failed FIPS (Attempt 1)


No structures returned for FIPS 46102
No structures returned for FIPS 02158
No structures returned for FIPS 02066

Retrying 3 failed FIPS (Attempt 2)


No structures returned for FIPS 46102
No structures returned for FIPS 02158
No structures returned for FIPS 02066

Retrying 3 failed FIPS (Attempt 3)


No structures returned for FIPS 46102
No structures returned for FIPS 02158
No structures returned for FIPS 02066
Final failed FIPS after 3 retries: ['46102', '02158', '02066']


In [9]:
# Turn into a dict for quick lookup
fire_to_fips = dict(zip(fire_fips_lookup["INCIDENT_ID"], fire_fips_lookup["GEOID"]))

In [10]:
# --- Force results into a geodataframe
nsi = gpd.GeoDataFrame(pd.concat(nsi_results, ignore_index=True), geometry="geometry", crs="EPSG:4326")
print(f"Retrieved {len(nsi)} NSI structure points.")

# --- Merge with existing FIPS data if it exists
out_fp = os.path.join(projdir,"data/spatial/nsi_structures.geojson")
if not os.path.exists(out_fp):
    print("No structure data found")
else:
    nsi_all = gpd.read_file(out_fp)
    print(len(nsi_all))
    nsi_all = pd.concat([nsi_all, nsi], ignore_index=True)
    print(len(nsi_all))

Retrieved 13070268 NSI structure points.
40757462
53827730
3222 NSI structures fall within fire perimeters.

Index(['geometry', 'fd_id', 'bid', 'occtype', 'st_damcat', 'bldgtype',
       'found_type', 'cbfips', 'pop2amu65', 'pop2amo65', 'pop2pmu65',
       'pop2pmo65', 'sqft', 'num_story', 'ftprntid', 'ftprntsrc', 'students',
       'found_ht', 'val_struct', 'val_cont', 'val_vehic', 'source',
       'med_yr_blt', 'firmzone', 'o65disable', 'u65disable', 'x', 'y',
       'ground_elv', 'ground_elv_m', 'FIPS', 'index_right', 'INCIDENT_ID'],
      dtype='object')



In [13]:
print(len(nsi_all))

53827730


In [15]:
nsi_all.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [16]:
# --- Now extract within fire perimeters
nsi_all = nsi_all.to_crs('EPSG:5070')
incis_ = incis_.to_crs('EPSG:5070')
# find structures within 500m of the fire perimeter
incis_perims = incis_.copy()
incis_perims['geometry'] = incis_perims.geometry.buffer(500)
nsi_fire = gpd.sjoin(nsi_all, incis_perims[["INCIDENT_ID", "geometry"]], how="inner", predicate="within")
print(f"{len(nsi_fire)} NSI structures fall within fire perimeters.")
print(f"\n{nsi_fire.columns}\n")

285625 NSI structures fall within fire perimeters.

Index(['fd_id', 'bid', 'occtype', 'st_damcat', 'bldgtype', 'found_type',
       'cbfips', 'pop2amu65', 'pop2amo65', 'pop2pmu65', 'pop2pmo65', 'sqft',
       'num_story', 'ftprntid', 'ftprntsrc', 'students', 'found_ht',
       'val_struct', 'val_cont', 'val_vehic', 'source', 'med_yr_blt',
       'firmzone', 'o65disable', 'u65disable', 'x', 'y', 'ground_elv',
       'ground_elv_m', 'FIPS', 'geometry', 'index_right', 'INCIDENT_ID'],
      dtype='object')



In [18]:
# --- save both files out:
out_fp = os.path.join(projdir,"data/spatial/mod/nsi/nsi_structures_in_fires_2014to2023.gpkg")
nsi_fire.to_file(out_fp, driver="GPKG")
print(f"Saved to {out_fp}")

# all structures (by county)
out_fp = os.path.join(projdir,"data/spatial/mod/nsi_structures_2014to2023.gpkg")
nsi_all.to_file(out_fp)
print(f"Saved to {out_fp}")

Saved to /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/spatial/mod/nsi/nsi_structures_in_fires_2014to2023.gpkg
Saved to /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/spatial/mod/nsi_structures_2014to2023.gpkg


In [ ]:
# # save a larger dataset within a buffer of fire perimeters
# buffer = inci_mtbs.copy()
# buffer['geometry'] = inci_mtbs.to_crs('EPSG:5070').geometry.buffer(10000) # make sure were in meters for the buffer
# buffer = buffer.to_crs('EPSG:4326') # and make to WGS for joining to NSI
# # perform the spatial join
# nsi_fire10k = gpd.sjoin(nsi, buffer[["MTBS_ID", "Incid_Name", "Ig_Date", "geometry"]], how="inner", predicate="within")
# print(f"{len(nsi_fire10k)} NSI structures are within 10km of fire perimeters.")
# print(f"\n{nsi_fire10k.columns}\n")
#
# # Save outputs
# out_fp = os.path.join(projdir,"data/spatial/nsi_structures_in_fires_50km.geojson")
# nsi_fire10k.to_file(out_fp, driver="GeoJSON")
# print(f"Save structure data to: {out_fp}")